# Notebook 04 — Variable Reduction

This notebook applies variable-selection procedures to the model-building sample established in Notebook 03.

Backward selection is first used to identify an approximate preferred model size. Models with nearby subset sizes are then examined using multiple model-selection criteria, including $R^2$, adjusted $R^2$, Mallows' $C_p$, AIC, BIC / SBC, and PRESS.

The objective is to identify a small set of defensible candidate specifications for subsequent diagnostic analysis and validation.

## 1. Import & Reconstruct Model-Building Sample

In [18]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

import statsmodels.formula.api as smf

In [19]:
PROJECT_DIR = Path.cwd().resolve()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = (
    PROCESSED_DATA_DIR
    / "db1c_ticket_prepared_202606.parquet"
)

df = pd.read_parquet(DATA_FILE)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows:    90,086
Columns: 16


### 1.1 Reproduce the 50/50 Model-Building Split

In [20]:
SPLIT_SEED = 42
TRAIN_SHARE = 0.50

rng = np.random.default_rng(SPLIT_SEED)

random_order = rng.permutation(len(df))
split_point = int(TRAIN_SHARE * len(df))

train_idx = random_order[:split_point]
validation_idx = random_order[split_point:]

model_df = (
    df.iloc[train_idx]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    df.iloc[validation_idx]
    .copy()
    .reset_index(drop=True)
)

print(f"Model-building sample: {len(model_df):,}")
print(f"Validation sample:     {len(validation_df):,}")

Model-building sample: 45,043
Validation sample:     45,043


### 1.2 Reproduce Categorical Grouping

In [21]:
MIN_CATEGORY_COUNT = 100

# Carrier grouping based only on model-building data
carrier_counts = model_df["RpCarrier"].value_counts()

major_carriers = carrier_counts[
    carrier_counts >= MIN_CATEGORY_COUNT
].index

model_df["RpCarrier_Model"] = model_df["RpCarrier"].where(
    model_df["RpCarrier"].isin(major_carriers),
    "Other"
)

validation_df["RpCarrier_Model"] = validation_df["RpCarrier"].where(
    validation_df["RpCarrier"].isin(major_carriers),
    "Other"
)

# Origin-state grouping based only on model-building data
state_counts = model_df["OriginState"].value_counts()

major_states = state_counts[
    state_counts >= MIN_CATEGORY_COUNT
].index

model_df["OriginState_Model"] = model_df["OriginState"].where(
    model_df["OriginState"].isin(major_states),
    "Other"
)

validation_df["OriginState_Model"] = validation_df["OriginState"].where(
    validation_df["OriginState"].isin(major_states),
    "Other"
)

print(
    f"Carrier model categories: "
    f"{model_df['RpCarrier_Model'].nunique()}"
)

print(
    f"Origin-state model categories: "
    f"{model_df['OriginState_Model'].nunique()}"
)

Carrier model categories: 12
Origin-state model categories: 43


#### Interpretation

Notebook 04 reproduces exactly the same model-building sample and categorical grouping rules used in Notebook 03.

The validation sample remains reserved and is not used for variable-selection decisions.

## 2. Define Full Model and Predictor Blocks
The variable-reduction analysis begins with the eight conceptual predictor blocks included in the broad model:

- `TotalDistance`
- `Coupons`
- `RoundTrip`
- `OnLine`
- `PurchaseWindowGroup`
- `RpCarrier_Model`
- `ItinGeoType`
- `OriginState_Model`

Categorical predictors are treated as complete blocks. If a categorical predictor is removed, all of its associated dummy variables are removed together.

### 2.1 Define Predictor Blocks

In [22]:
predictor_terms = {
    "TotalDistance":
        "TotalDistance",

    "Coupons":
        "Coupons",

    "RoundTrip":
        "C(RoundTrip, Treatment(reference=0))",

    "OnLine":
        "C(OnLine, Treatment(reference=1))",

    "PurchaseWindowGroup":
        "C(PurchaseWindowGroup, Treatment(reference='21AP'))",

    "RpCarrier_Model":
        "C(RpCarrier_Model, Treatment(reference='AA'))",

    "ItinGeoType":
        "C(ItinGeoType, Treatment(reference=2))",

    "OriginState_Model":
        "C(OriginState_Model, Treatment(reference='CA'))"
}

print("Predictor blocks:")
for name in predictor_terms:
    print(f"- {name}")

Predictor blocks:
- TotalDistance
- Coupons
- RoundTrip
- OnLine
- PurchaseWindowGroup
- RpCarrier_Model
- ItinGeoType
- OriginState_Model


### 2.2 Fit the Full Starting Model

In [23]:
full_formula = (
    "TotalAmount ~ "
    + " + ".join(predictor_terms.values())
)

full_model = smf.ols(
    formula=full_formula,
    data=model_df
).fit()

print(f"Observations:        {int(full_model.nobs):,}")
print(f"Model df:            {int(full_model.df_model)}")
print(f"R-squared:           {full_model.rsquared:.4f}")
print(f"Adjusted R-squared:  {full_model.rsquared_adj:.4f}")
print(f"SSE:                 {full_model.ssr:.2f}")
print(f"MSE:                 {full_model.mse_resid:.4f}")
print(f"AIC:                 {full_model.aic:.2f}")
print(f"BIC:                 {full_model.bic:.2f}")

Observations:        45,043
Model df:            60
R-squared:           0.3854
Adjusted R-squared:  0.3846
SSE:                 3434705201.20
MSE:                 76357.3252
AIC:                 634313.98
BIC:                 634845.62


## 3. Backward Selection

Backward selection is used as the automatic procedure for identifying an approximate appropriate model size.

The procedure starts from the full eight-block model. At each step, each remaining predictor block is removed in turn and the reduced model is compared with the current model using a partial F-test.

The predictor with the largest non-significant partial-F p-value is removed.

The procedure continues until every remaining predictor block makes a statistically significant incremental contribution at the 5% level.

### 3.1 Backward-Selection Procedure

In [24]:
ALPHA = 0.05

current_predictors = list(predictor_terms.keys())
backward_history = []

step = 1

while True:

    current_formula = (
        "TotalAmount ~ "
        + " + ".join(
            predictor_terms[name]
            for name in current_predictors
        )
    )

    current_model = smf.ols(
        formula=current_formula,
        data=model_df
    ).fit()

    removal_tests = []

    for predictor in current_predictors:

        reduced_predictors = [
            name
            for name in current_predictors
            if name != predictor
        ]

        reduced_formula = (
            "TotalAmount ~ "
            + " + ".join(
                predictor_terms[name]
                for name in reduced_predictors
            )
        )

        reduced_model = smf.ols(
            formula=reduced_formula,
            data=model_df
        ).fit()

        f_stat, p_value, df_diff = (
            current_model.compare_f_test(reduced_model)
        )

        removal_tests.append({
            "Step": step,
            "Predictor": predictor,
            "Partial_F": f_stat,
            "p_value": p_value,
            "df_removed": int(df_diff)
        })

    step_table = (
        pd.DataFrame(removal_tests)
        .sort_values("p_value", ascending=False)
        .reset_index(drop=True)
    )

    backward_history.append(step_table)

    print(f"\nBackward-selection step {step}")
    display(step_table)

    worst_predictor = step_table.loc[0, "Predictor"]
    worst_p_value = step_table.loc[0, "p_value"]

    if worst_p_value <= ALPHA:
        break

    print(
        f"Removing {worst_predictor} "
        f"(p = {worst_p_value:.4f})"
    )

    current_predictors.remove(worst_predictor)

    step += 1


Backward-selection step 1


,Step,Predictor,Partial_F,p_value,df_removed
0,1,Coupons,2.815292,9.337688e-02,1
1,1,ItinGeoType,45.415674,1.612706e-11,1
2,1,OnLine,177.807373,1.742369e-40,1
3,1,OriginState_Model,10.078394,4.023863e-64,42
4,1,RoundTrip,1553.964202,0.000000e+00,1
5,1,TotalDistance,5258.104567,0.000000e+00,1
6,1,RpCarrier_Model,430.710030,0.000000e+00,11
7,1,PurchaseWindowGroup,1187.846001,0.000000e+00,2


Removing Coupons (p = 0.0934)

Backward-selection step 2


,Step,Predictor,Partial_F,p_value,df_removed
0,2,ItinGeoType,44.473225,2.608356e-11,1
1,2,OnLine,266.838069,8.251201e-60,1
2,2,OriginState_Model,10.222520,2.654990e-65,42
3,2,RoundTrip,2352.571503,0.000000e+00,1
4,2,TotalDistance,6653.021424,0.000000e+00,1
5,2,RpCarrier_Model,436.842642,0.000000e+00,11
6,2,PurchaseWindowGroup,1189.718462,0.000000e+00,2


In [25]:
selected_predictors = current_predictors.copy()

print("\nFinal predictors retained:")
for predictor in selected_predictors:
    print(f"- {predictor}")

print(
    f"\nNumber of retained predictor blocks: "
    f"{len(selected_predictors)}"
)


Final predictors retained:
- TotalDistance
- RoundTrip
- OnLine
- PurchaseWindowGroup
- RpCarrier_Model
- ItinGeoType
- OriginState_Model

Number of retained predictor blocks: 7


### 3.2 Backward-Selection Result

Backward selection removes `Coupons` at the first step:

$$
F = 2.815,
\qquad
p = 0.0934.
$$

Because the partial F-test is not statistically significant at the 5% level, `Coupons` does not provide sufficient incremental explanatory value conditional on the other predictors.

The model is then re-estimated without `Coupons`. At the second step, all seven remaining predictor blocks have statistically significant partial F-tests.

The backward-selection procedure therefore terminates with:

$$
\boxed{7\text{ predictor blocks}}
$$

The retained predictors are:

- `TotalDistance`
- `RoundTrip`
- `OnLine`
- `PurchaseWindowGroup`
- `RpCarrier_Model`
- `ItinGeoType`
- `OriginState_Model`

This seven-block result is treated as an approximate preferred model size rather than being accepted automatically as the final specification.

In [26]:
backward_history_table = pd.concat(
    backward_history,
    ignore_index=True
)

backward_history_table.to_csv(
    OUTPUT_DIR / "04_backward_selection_history.csv",
    index=False
)

backward_selected = pd.DataFrame({
    "Predictor": selected_predictors
})

backward_selected.to_csv(
    OUTPUT_DIR / "04_backward_selected_predictors.csv",
    index=False
)

## 4. Nearby-Size Subset Search

The backward-selection procedure identifies seven predictor blocks as the approximate preferred model size.

Following the modelling approach discussed in class, the automatically selected model is not accepted mechanically as the final specification.

Instead, all candidate models with nearby subset sizes are examined:

$$
6,\qquad 7,\qquad 8
$$

predictor blocks.

With eight available predictor blocks, this gives:

$$
\binom{8}{6}
+
\binom{8}{7}
+
\binom{8}{8}
=
28+8+1
=
37
$$

candidate models.

These models are compared using:

$$
R^2,\quad
R^2_{\text{adj}},\quad
C_p,\quad
AIC,\quad
BIC/SBC,\quad
PRESS.
$$

The objective is to determine whether another six-, seven-, or eight-block specification provides a better balance between explanatory performance and model parsimony than the model returned automatically by backward selection.

### 4.1 Estimate All 6-, 7-, and 8-Block Models

In [27]:
subset_results = []

full_mse = full_model.mse_resid
n = int(full_model.nobs)

predictor_names = list(predictor_terms.keys())

for r in [6, 7, 8]:

    for subset in combinations(predictor_names, r):

        terms = [
            predictor_terms[name]
            for name in subset
        ]

        formula = (
            "TotalAmount ~ "
            + " + ".join(terms)
        )

        model = smf.ols(
            formula=formula,
            data=model_df
        ).fit()

        # Actual number of estimated parameters,
        # including the intercept
        p = int(model.df_model + 1)

        # Mallows' Cp
        cp = (
            model.ssr / full_mse
            - (n - 2 * p)
        )

        # Exact leave-one-out PRESS
        influence = model.get_influence()
        leverage = influence.hat_matrix_diag

        press = np.sum(
            (
                model.resid
                / (1 - leverage)
            ) ** 2
        )

        subset_results.append({
            "Predictors": subset,
            "Num_Blocks": len(subset),
            "Num_Parameters": p,
            "R2": model.rsquared,
            "Adj_R2": model.rsquared_adj,
            "SSE": model.ssr,
            "MSE": model.mse_resid,
            "Cp": cp,
            "AIC": model.aic,
            "BIC": model.bic,
            "PRESS": press
        })

subset_table = pd.DataFrame(subset_results)

print(
    f"Number of candidate models: "
    f"{len(subset_table)}"
)

print("\nModels by subset size:")

display(
    subset_table["Num_Blocks"]
    .value_counts()
    .sort_index()
    .rename("Number of Models")
    .to_frame()
)

Number of candidate models: 37

Models by subset size:


,Number of Models
Num_Blocks,
6,28
7,8
8,1


### 4.2 Rank Candidate Models by Selection Criteria

The 37 candidate models are compared using the model-selection criteria discussed in class.

The preferred direction for each criterion is:

- $R^2$: larger is better;
- adjusted $R^2$: larger is better;
- Mallows' $C_p$: relatively small values close to the number of estimated parameters $p$ are preferred;
- AIC: smaller is better;
- BIC / SBC: smaller is better;
- PRESS: smaller is better.

Because $R^2$ cannot decrease when additional predictors are added, it is interpreted together with criteria that explicitly account for model complexity.

In [28]:
subset_table["Cp_minus_p"] = (
    subset_table["Cp"]
    - subset_table["Num_Parameters"]
)

subset_table["Abs_Cp_minus_p"] = (
    subset_table["Cp_minus_p"].abs()
)

In [29]:
top_r2 = (
    subset_table
    .sort_values("R2", ascending=False)
    .head(5)
)

top_adj_r2 = (
    subset_table
    .sort_values("Adj_R2", ascending=False)
    .head(5)
)

top_cp = (
    subset_table
    .sort_values(
        ["Abs_Cp_minus_p", "Cp"],
        ascending=[True, True]
    )
    .head(5)
)

top_aic = (
    subset_table
    .sort_values("AIC")
    .head(5)
)

top_bic = (
    subset_table
    .sort_values("BIC")
    .head(5)
)

top_press = (
    subset_table
    .sort_values("PRESS")
    .head(5)
)

In [30]:
display_cols = [
    "Predictors",
    "Num_Blocks",
    "Num_Parameters",
    "R2",
    "Adj_R2",
    "Cp",
    "Cp_minus_p",
    "AIC",
    "BIC",
    "PRESS"
]

print("Top 5 by R-squared")
display(
    top_r2[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by Adjusted R-squared")
display(
    top_adj_r2[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by Mallows' Cp")
display(
    top_cp[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by AIC")
display(
    top_aic[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by BIC / SBC")
display(
    top_bic[display_cols]
    .reset_index(drop=True)
)

print("\nTop 5 by PRESS")
display(
    top_press[display_cols]
    .reset_index(drop=True)
)

Top 5 by R-squared


,Predictors,Num_Blocks,Num_Parameters,R2,Adj_R2,Cp,Cp_minus_p,AIC,BIC,PRESS
0,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",8,61,0.385439,0.384619,61.000000,0.000000,634313.978234,634845.615979,3.443909e+09
1,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",7,60,0.385401,0.384595,61.815292,1.815292,634314.797255,634837.719627,3.443877e+09
2,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",7,60,0.384819,0.384012,104.415674,44.415674,634357.432553,634880.354925,3.447039e+09
3,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",6,59,0.384793,0.384000,104.290311,45.290311,634357.307801,634871.514800,3.446935e+09
4,"(TotalDistance, Coupons, RoundTrip, PurchaseWi...",7,60,0.383010,0.382201,236.807373,176.807373,634489.675755,635012.598127,3.457297e+09



Top 5 by Adjusted R-squared


,Predictors,Num_Blocks,Num_Parameters,R2,Adj_R2,Cp,Cp_minus_p,AIC,BIC,PRESS
0,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",8,61,0.385439,0.384619,61.000000,0.000000,634313.978234,634845.615979,3.443909e+09
1,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",7,60,0.385401,0.384595,61.815292,1.815292,634314.797255,634837.719627,3.443877e+09
2,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",7,60,0.384819,0.384012,104.415674,44.415674,634357.432553,634880.354925,3.447039e+09
3,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",6,59,0.384793,0.384000,104.290311,45.290311,634357.307801,634871.514800,3.446935e+09
4,"(TotalDistance, Coupons, RoundTrip, PurchaseWi...",7,60,0.383010,0.382201,236.807373,176.807373,634489.675755,635012.598127,3.457297e+09



Top 5 by Mallows' Cp


...

,Predictors,Num_Blocks,Num_Parameters,R2,Adj_R2,Cp,Cp_minus_p,AIC,BIC,PRESS
0,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",8,61,0.385439,0.384619,61.000000,0.000000,634313.978234,634845.615979,3.443909e+09
1,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",7,60,0.385401,0.384595,61.815292,1.815292,634314.797255,634837.719627,3.443877e+09
2,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",6,59,0.384793,0.384000,104.290311,45.290311,634357.307801,634871.514800,3.446935e+09
3,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",7,60,0.384819,0.384012,104.415674,44.415674,634357.432553,634880.354925,3.447039e+09
4,"(TotalDistance, Coupons, RoundTrip, PurchaseWi...",7,60,0.383010,0.382201,236.807373,176.807373,634489.675755,635012.598127,3.457297e+09



Top 5 by BIC / SBC


,Predictors,Num_Blocks,Num_Parameters,R2,Adj_R2,Cp,Cp_minus_p,AIC,BIC,PRESS
0,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",6,18,0.379576,0.379341,404.174012,386.174012,634655.697041,634812.573752,3.470603e+09
1,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",6,18,0.379535,0.379300,407.178444,389.178444,634658.677015,634815.553727,3.470821e+09
2,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",7,19,0.379656,0.379408,400.292529,381.292529,634651.862864,634817.454949,3.470398e+09
3,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",7,60,0.385401,0.384595,61.815292,1.815292,634314.797255,634837.719627,3.443877e+09
4,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",8,61,0.385439,0.384619,61.000000,0.000000,634313.978234,634845.615979,3.443909e+09



Top 5 by PRESS


,Predictors,Num_Blocks,Num_Parameters,R2,Adj_R2,Cp,Cp_minus_p,AIC,BIC,PRESS
0,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",7,60,0.385401,0.384595,61.815292,1.815292,634314.797255,634837.719627,3.443877e+09
1,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",8,61,0.385439,0.384619,61.000000,0.000000,634313.978234,634845.615979,3.443909e+09
2,"(TotalDistance, RoundTrip, OnLine, PurchaseWin...",6,59,0.384793,0.384000,104.290311,45.290311,634357.307801,634871.514800,3.446935e+09
3,"(TotalDistance, Coupons, RoundTrip, OnLine, Pu...",7,60,0.384819,0.384012,104.415674,44.415674,634357.432553,634880.354925,3.447039e+09
4,"(TotalDistance, Coupons, RoundTrip, PurchaseWi...",7,60,0.383010,0.382201,236.807373,176.807373,634489.675755,635012.598127,3.457297e+09


In [31]:
subset_table_export = subset_table.copy()

subset_table_export["Predictors"] = (
    subset_table_export["Predictors"]
    .apply(lambda x: ", ".join(x))
)

subset_table_export.to_csv(
    OUTPUT_DIR / "04_nearby_subset_model_rankings.csv",
    index=False
)

### 4.3 Criterion Winners and Candidate Models

In [32]:
criterion_winners = pd.DataFrame([
    {
        "Criterion": "R-squared",
        **top_r2.iloc[0].to_dict()
    },
    {
        "Criterion": "Adjusted R-squared",
        **top_adj_r2.iloc[0].to_dict()
    },
    {
        "Criterion": "Mallows' Cp",
        **top_cp.iloc[0].to_dict()
    },
    {
        "Criterion": "AIC",
        **top_aic.iloc[0].to_dict()
    },
    {
        "Criterion": "BIC / SBC",
        **top_bic.iloc[0].to_dict()
    },
    {
        "Criterion": "PRESS",
        **top_press.iloc[0].to_dict()
    }
])

criterion_winners["Predictors"] = (
    criterion_winners["Predictors"]
    .apply(lambda x: ", ".join(x))
)

winner_cols = [
    "Criterion",
    "Predictors",
    "Num_Blocks",
    "Num_Parameters",
    "R2",
    "Adj_R2",
    "Cp",
    "AIC",
    "BIC",
    "PRESS"
]

criterion_winners[winner_cols].to_csv(
    OUTPUT_DIR / "04_criterion_winners.csv",
    index=False
)

display(
    criterion_winners[winner_cols]
)

,Criterion,Predictors,Num_Blocks,Num_Parameters,R2,Adj_R2,Cp,AIC,BIC,PRESS
0,R-squared,"TotalDistance, Coupons, RoundTrip, OnLine, Pur...",8,61,0.385439,0.384619,61.000000,634313.978234,634845.615979,3.443909e+09
1,Adjusted R-squared,"TotalDistance, Coupons, RoundTrip, OnLine, Pur...",8,61,0.385439,0.384619,61.000000,634313.978234,634845.615979,3.443909e+09
2,Mallows' Cp,"TotalDistance, Coupons, RoundTrip, OnLine, Pur...",8,61,0.385439,0.384619,61.000000,634313.978234,634845.615979,3.443909e+09
3,AIC,"TotalDistance, Coupons, RoundTrip, OnLine, Pur...",8,61,0.385439,0.384619,61.000000,634313.978234,634845.615979,3.443909e+09
4,BIC / SBC,"TotalDistance, Coupons, RoundTrip, OnLine, Pur...",6,18,0.379576,0.379341,404.174012,634655.697041,634812.573752,3.470603e+09
5,PRESS,"TotalDistance, RoundTrip, OnLine, PurchaseWind...",7,60,0.385401,0.384595,61.815292,634314.797255,634837.719627,3.443877e+09


The nearby-size subset search does not identify a single model that is preferred by every selection criterion.

Because the objective of this project is primarily explanatory rather than purely predictive, greater weight is placed on adjusted $R^2$, Mallows' $C_p$, interpretability, and the incremental contribution of predictor blocks. PRESS is also considered because it provides a measure of predictive performance.

Two specifications emerge as the strongest overall candidates.

**Candidate 1 — Full eight-block model**

The full model is preferred by:

- $R^2$;
- adjusted $R^2$;
- Mallows' $C_p$;
- AIC.

For the full model,

$$
C_p = p = 61,
$$

which follows from its role as the reference model used to estimate the error variance in the Mallows' $C_p$ calculation. It therefore provides the benchmark against which the subset models are assessed.

**Candidate 2 — Backward-selected seven-block model**

The backward-selected model removes `Coupons` and is preferred by PRESS.

Its Mallows' $C_p$ result is also favourable:

$$
C_p \approx 61.82,
\qquad
p = 60.
$$

The seven-block model therefore achieves almost the same explanatory and predictive performance as the full model with one fewer predictor block.

BIC / SBC instead favours a substantially smaller six-block specification containing:

- `TotalDistance`
- `Coupons`
- `RoundTrip`
- `OnLine`
- `PurchaseWindowGroup`
- `RpCarrier_Model`

This model has only 18 estimated parameters because it excludes both `ItinGeoType` and the high-dimensional `OriginState_Model` block.

However:

$$
C_p \approx 404.17,
\qquad
p = 18,
$$

which is much less favourable than the two leading models, and the model also sacrifices more explanatory fit.

The BIC-selected model is therefore interpreted as evidence of the stronger parsimony preference imposed by BIC / SBC, but it is not retained as a primary candidate for this explanatory modelling objective.

Accordingly, no final model is selected at this stage. The full eight-block model and the backward-selected seven-block model are carried forward for diagnostic analysis and later validation using the reserved validation sample.

## 5. Variable-Reduction Summary

Backward selection first identified a seven-block model by removing `Coupons`.

Following the modelling approach discussed in class, the automatic result was not accepted mechanically. All 6-, 7-, and 8-block specifications were therefore examined, giving 37 nearby-size candidate models.

The comparison across $R^2$, adjusted $R^2$, Mallows' $C_p$, AIC, BIC / SBC, and PRESS identifies two main candidates:

1. the full eight-block model;
2. the seven-block model without `Coupons`.

The full model performs best on most explanatory criteria, while the seven-block model provides almost identical performance with slightly greater parsimony and the lowest PRESS.

No final model is selected at this stage. Both candidates are carried forward to model diagnostics and later validation.

In [33]:
candidate_models = pd.DataFrame({
    "Candidate": [
        "Full 8-block model",
        "Backward 7-block model"
    ],
    "Predictors": [
        (
            "TotalDistance, Coupons, RoundTrip, OnLine, "
            "PurchaseWindowGroup, RpCarrier_Model, "
            "ItinGeoType, OriginState_Model"
        ),
        (
            "TotalDistance, RoundTrip, OnLine, "
            "PurchaseWindowGroup, RpCarrier_Model, "
            "ItinGeoType, OriginState_Model"
        )
    ]
})

candidate_models.to_csv(
    OUTPUT_DIR / "04_candidate_models.csv",
    index=False
)

display(candidate_models)

,Candidate,Predictors
0,Full 8-block model,"TotalDistance, Coupons, RoundTrip, OnLine, Pur..."
1,Backward 7-block model,"TotalDistance, RoundTrip, OnLine, PurchaseWind..."
